In [1]:
import numpy as np
import matplotlib.pyplot as plt
#import lattpy  as lp
import imageio
from matplotlib.gridspec import GridSpec
from tqdm import tqdm

In [2]:
#Generate different images for different lattice constants
#a_list = np.linspace(100, 1, 5) # list of lattice constants
#lattices_plots = [] # empty list for storaging
#for a_c in a_list:
#    latt = lp.simple_cubic(a=a_c, neighbors=1)
#   s = latt.build(shape = (5, 5, 5))
#    center_point = latt.center()
#    plot = latt.plot(show=False)
#    fig = plot.get_figure()
#    limit = 5
#    plot.set_xlim(-limit, limit)
#    plot.set_ylim(-limit, limit)  
#    plot.set_zlim(-limit, limit)
    # Convert image for imagio
#    fig.canvas.draw()
#    img = np.frombuffer(fig.canvas.tostring_rgb(), dtype=np.uint8)
#    img = img.reshape(fig.canvas.get_width_height()[::-1] + (3,))
#    lattices_plots.append(img)

#    plt.close(fig)

In [3]:
#Create gif
#imageio.mimsave('lattice_plots.gif', lattices_plots, duration = 2)

In [4]:
#I do not like the package lattpy....

In [5]:
# Function to create list representing a simple plane
def simple_plane(a, xlim, ylim): # a: lattice constant, xlim, ylim: limits
    data = []
    nx = int(2*xlim / a) + 1
    ny = int(2*ylim / a) + 1
    for i in range(nx):
        for j in range(ny):
            x = i * a -xlim
            y = j * a -ylim
            data.append([x, y])
    return data

def simple_cubic(a, xlim, ylim, zlim):  # a: lattice constant, xlim, ylim, zlim: limits
    data = []
    nz = int(2*zlim / a) + 1
    plane = simple_plane(a, xlim, ylim) #create simple plane

    for k in range(nz):
        z = k * a -zlim

        for point in plane:
            x, y = point
            data.append([x,y,z])
    
    return data

#Plot function for simple cubic
def cubic_plot(cube, xlim, ylim, zlim, color):
    fig = plt.figure()
    ax = fig.add_subplot(111, projection = '3d')
    xs = [p[0] for p in cube]
    ys = [p[1] for p in cube]
    zs = [p[2] for p in cube]
    ax.scatter(xs, ys, zs, c=color, marker='o') 
    ax.set_xlabel('X')
    ax.set_ylabel('Y')
    ax.set_zlabel('Z')
    ax.set_title(f'Real space plot')

    offset_x = 0.1 * xlim
    offset_y = 0.1 * ylim
    offset_z = 0.1 * zlim

    ax.set_xlim(-xlim - offset_x, xlim + offset_x)
    ax.set_ylim(-ylim - offset_y, ylim + offset_y)
    ax.set_zlim(-zlim - offset_z, zlim + offset_z)

    ax.view_init(elev=30, azim=30)

    return fig

In [6]:
# FT calculations
def ft_cube(cube, kmax, n_k=None):
    cube = np.array(cube)
    if n_k == None:
        nkx = len(np.unique(cube[:,0]))
        nky = len(np.unique(cube[:,1]))
        nkz = len(np.unique(cube[:,2]))
    else:
        nkx = n_k
        nky = n_k
        nkz = n_k

    kx = np.linspace(-kmax, kmax, nkx)
    ky = np.linspace(-kmax, kmax, nky)
    kz = np.linspace(-kmax, kmax, nkz)

    # Calculate
    data = [] #Storage

    for kxi in kx:
        for kyj in ky:
            for kzk in kz:
                k_vec = [kxi, kyj, kzk]
                data_point = np.sum(np.exp(-2j*np.pi*np.dot(cube, k_vec)))
                data.append(data_point)
    intensity = np.abs(data)**2
    phase = np.angle(data)

    return intensity, phase, kx, ky, kz

# Plot function for ft attribute (phase or intensity)
def cubic_plot_ft(attribute, kx, ky, kz, attribute_name):
    fig = plt.figure()
    ax = fig.add_subplot(111, projection = '3d')

    ax.set_xlabel('KX')
    ax.set_ylabel('KY')
    ax.set_zlabel('KZ')
    ax.set_title(f'FT plot ({attribute_name})')
    KX, KY, KZ = np.meshgrid(kx, ky, kz, indexing='ij')
    kx_f = KX.flatten()
    ky_f = KY.flatten()
    kz_f = KZ.flatten()

    colors_attribute = attribute.flatten()
    offset_kx = 0.1 * np.max(np.abs(kx_f))
    offset_ky = 0.1 * np.max(np.abs(ky_f))
    offset_kz = 0.1 * np.max(np.abs(kz_f))

    ax.set_xlim(-np.max(kx_f) - offset_kx, np.max(kx_f) + offset_kx)
    ax.set_ylim(-np.max(ky_f) - offset_ky, np.max(ky_f) + offset_ky)
    ax.set_zlim(-np.max(kz_f) - offset_kz, np.max(kz_f) + offset_kz)
    sc = ax.scatter(kx_f, ky_f, kz_f, c=colors_attribute, cmap='plasma')
    fig.colorbar(sc, ax=ax, shrink = 0.2, label = attribute_name)
    ax.view_init(elev=30, azim=30)
    return fig



In [7]:
#Turn the figure into an image
def fig_to_img(fig):
    fig.canvas.draw()
    img = np.frombuffer(fig.canvas.tostring_rgb(), dtype=np.uint8)
    img = img.reshape(fig.canvas.get_width_height()[::-1] + (3,))
    return img

In [8]:
#Generate different images for different lattice constants
a_list = np.linspace(30, 9, 40) # list of lattice constants
lattices_plots = [] # empty list for storaging
xlim = 50
ylim = 50
zlim = 50

a_min = np.min(a_list)
kmax = 2*np.pi / a_min
a_max = np.max(a_list) 
dk = (2*np.pi) / a_max / 5
n_k = int(2 * kmax / dk +1) # for constant sampling in Fourier Space
if n_k % 2 == 0:
    n_k += 1

for a_c in tqdm(a_list):
    #Create real cube
    latt = simple_cubic(a=a_c, xlim=xlim, ylim=ylim, zlim=zlim)
    plot_r = cubic_plot(latt, xlim, ylim, zlim, 'b') # real space plot
    fig_r = plot_r.get_figure() 

    #Create FT of real cube
    k = 1/a_c
    kmax = 2*np.pi / a_min  
    n_k = int(2*kmax / dk + 1)
    n_k = 51
    n_k = int(np.ceil(10*2* kmax / (2*np.pi/a_c))) | 1  
    intensity, phase, kx, ky, kz = ft_cube(latt, kmax, n_k)
    plot_ft_i = cubic_plot_ft(intensity, kx, ky, kz, 'Intensity') # ft intensity plot 
    fig_ft_i = plot_ft_i.get_figure() 
    plot_ft_p = cubic_plot_ft(phase, kx, ky, kz, 'Phase') # ft phase plot 
    fig_ft_p = plot_ft_p.get_figure() 

    #Put them all together
    img_r = fig_to_img(fig_r)
    img_ft_i = fig_to_img(fig_ft_i)
    img_ft_p = fig_to_img(fig_ft_p)
    plt.close(fig_r)
    plt.close(fig_ft_i)
    plt.close(fig_ft_p)

    fig_master = plt.figure()
    gs = GridSpec(2, 2, figure=fig_master, width_ratios=[1,1], height_ratios=[1,1])

    # Real spacus
    ax_r = fig_master.add_subplot(gs[0,0])
    ax_r.imshow(img_r)
    ax_r.axis('off')


    # FT int
    ax_i = fig_master.add_subplot(gs[0,1])
    ax_i.imshow(img_ft_i)
    ax_i.axis('off')

    # FT phasus
    ax_p = fig_master.add_subplot(gs[1,1])
    ax_p.imshow(img_ft_p)
    ax_p.axis('off')

    # Get 2D slice
    intensity = intensity.reshape(len(kx), len(ky), len(kz))
    phase = phase.reshape(len(kx), len(ky), len(kz))
    kz0_idx = np.argmin(np.abs(kz)) 
    intensity_2d = intensity[:, :, kz0_idx]

    fig2d = plt.figure()
    plt.imshow(intensity_2d, extent=[kx[0], kx[-1], ky[0], ky[-1]], origin='lower', cmap='plasma')
    plt.xlabel('kx')
    plt.ylabel('ky')
    plt.title(f'FT Intensity 2D slice (kz=0)')
    plt.colorbar(label='Intensity')
    img_s = fig_to_img(fig2d)
    plt.close(fig2d)

    ax_s = fig_master.add_subplot(gs[1,0])
    ax_s.imshow(img_s)
    ax_s.axis('off')
    fig_master.suptitle(f'Lattice constant a = {a_c:.2f}', fontsize=16)

    lattices_plots.append(fig_to_img(fig_master))

    plt.close(fig_master)



  0%|          | 0/40 [00:00<?, ?it/s]C:\Users\magda\AppData\Local\Temp\ipykernel_26180\1350254293.py:4: MatplotlibDeprecationWarning: The tostring_rgb function was deprecated in Matplotlib 3.8 and will be removed in 3.10. Use buffer_rgba instead.
  img = np.frombuffer(fig.canvas.tostring_rgb(), dtype=np.uint8)
100%|██████████| 40/40 [15:18<00:00, 22.96s/it]


In [9]:
#Create gifs
imageio.mimsave('lattice_plots.gif', lattices_plots, fps=3)

In [10]:
# Using FFT